# 01 — Exploratory Data Analysis
Pokémon TCG card prices. Run `make data` first so `data/processed/cards.parquet` exists.

Educational project — not financial advice.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, resolve_path, set_seeds
from src.features.build_features import build_task_a_frame

cfg = load_config()
set_seeds(cfg["seed"])
df = pd.read_parquet(resolve_path(cfg, "dataset"))
print(df.shape)
df.head()

In [ ]:
# Coverage: rows per snapshot, cards per variant
print(df.groupby("snapshot_date").size())
print(df["variant"].value_counts())

In [ ]:
# Price distribution is heavily right-skewed -> we model log price
frame = build_task_a_frame(df)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
frame["price_market"].clip(upper=200).hist(bins=80, ax=axes[0])
axes[0].set_title("market price (clipped at $200)")
frame["log_price"].hist(bins=80, ax=axes[1])
axes[1].set_title("log(market price)")
plt.show()

In [ ]:
# Median price by rarity — the strongest single driver
(frame.groupby("rarity")["price_market"]
      .agg(["median", "count"])
      .query("count >= 30")
      .sort_values("median", ascending=False)
      .head(20))

In [ ]:
# Price vs days since release, and set-level medians
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(frame["days_since_release"], frame["log_price"], s=3, alpha=0.2)
axes[0].set_xlabel("days since release"); axes[0].set_ylabel("log price")
set_med = frame.groupby("set_name")["price_market"].median().sort_values().tail(20)
set_med.plot.barh(ax=axes[1]); axes[1].set_title("priciest sets (median)")
plt.tight_layout(); plt.show()

In [ ]:
# Missingness check for feature columns
feat_cols = cfg["features"]["numeric"] + cfg["features"]["categorical"]
frame[[c for c in feat_cols if c in frame.columns]].isna().mean().sort_values(ascending=False)